<a href="https://colab.research.google.com/github/Frankieche24/White-Wine-Quality/blob/main/ML_Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning a Pre-trained CNN on MNIST

This notebook demonstrates how to fine-tune a pre-trained ResNet18 model for the MNIST dataset. It includes steps for data loading, model modification, training, and saving the trained model weights.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

# Define the device to run on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### 1. Define Hyperparameters

In [2]:
num_epochs = 2  # As requested, fine-tune for a couple of epochs
batch_size = 64
learning_rate = 0.001
num_classes = 10 # MNIST has 10 classes (digits 0-9)

### 2. Data Loading and Preprocessing

MNIST images are grayscale, but most pre-trained CNNs (like ResNet) expect 3-channel (RGB) input. We will transform the grayscale image to 3 channels by repeating the single channel three times.

In [3]:
# Define transformations for the MNIST dataset
transform = transforms.Compose([
    transforms.Resize(224), # Resize images to 224x224 for ResNet
    transforms.Grayscale(num_output_channels=3), # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet normalization
])

# Download and load the MNIST training dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Download and load the MNIST test dataset
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training data samples: {len(train_dataset)}")
print(f"Test data samples: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 479kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.3MB/s]

Training data samples: 60000
Test data samples: 10000


### 3. Model Definition

We will load a pre-trained ResNet18 model and modify its final fully connected layer to output 10 classes (for MNIST).

In [4]:
# Load a pre-trained ResNet18 model
model = models.resnet18(pretrained=True)

# Freeze all parameters in the network (optional, but common for fine-tuning)
# for param in model.parameters():
#     param.requires_grad = False

# Get the number of features in the last fully connected layer
num_ftrs = model.fc.in_features

# Replace the last fully connected layer with a new one for MNIST classes
model.fc = nn.Linear(num_ftrs, num_classes)

# Move the model to the specified device
model = model.to(device)

print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### 4. Loss Function and Optimizer

In [5]:
# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### 5. Training Loop

In [ ]:
print("Starting training...")

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Use tqdm for a progress bar
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_accuracy = correct_predictions / total_samples

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

print("Training finished.")

Starting training...


Epoch 1/2: 100%|██████████| 938/938 [04:43<00:00,  3.31it/s]


Epoch [1/2], Loss: 0.0669, Accuracy: 0.9799


Epoch 2/2:  81%|████████  | 759/938 [03:50<00:53,  3.33it/s]

### 6. Evaluation (Optional)

In [ ]:
print("Starting evaluation...")
model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculation during evaluation
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the {len(test_dataset)} test images: {100 * correct / total:.2f}%')
print("Evaluation finished.")

### 7. Save Model Weights

Finally, save the state dictionary of the trained model to a `.pth` file.

In [ ]:
model_save_path = 'mnist_resnet18_finetuned.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

### 8. Create `requirements.txt` for Deployment

This file lists all the Python packages required to run your application. This is essential for platforms like Hugging Face Spaces to set up the correct environment.

In [11]:
%%writefile requirements.txt
torch
torchvision
gradio>=6.0.0

Overwriting requirements.txt


### 9. Create `app.py` for Hugging Face Spaces

This file defines the Gradio application that will load your trained model and provide an interactive interface for users to test it. It includes the necessary imports, model loading, data preprocessing, prediction logic, and the Gradio interface setup.

In [15]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image, ImageOps
import gradio as gr

# ── Configuration ─────────────────────────────────────────────────
NUM_CLASSES    = 10
MODEL_FILENAME = "mnist_resnet18_finetuned.pth" # Corrected filename

# ── Model loading ─────────────────────────────────────────────────
def load_model(path: str = MODEL_FILENAME) -> nn.Module:
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location=torch.device("cpu")))
    m.eval()
    return m

model = load_model()

# ── Preprocessing ─────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Inference ─────────────────────────────────────────────────────
def predict(image) -> dict:
    if image is None:
        return {}
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = ImageOps.grayscale(image)
    tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1).squeeze()
    return {str(i): float(probs[i]) for i in range(NUM_CLASSES)}

def predict_sketch(sketch) -> dict:
    """
    gr.Sketchpad in Gradio 6 returns a dict:
      {"background": PIL.Image, "layers": [PIL.Image, ...], "composite": PIL.Image}
    We use the composite layer and INVERT it because the sketchpad
    draws dark strokes on a white background, but MNIST is white on black.
    """
    if sketch is None:
        return {}

    # Extract composite image (drawing merged with background)
    if isinstance(sketch, dict):
        image = sketch.get("composite") or sketch.get("background")
    else:
        image = sketch

    if image is None:
        return {}

    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)

    # Convert to grayscale then INVERT so digit is white on black (MNIST style)
    image = ImageOps.grayscale(image)
    image = ImageOps.invert(image)

    tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1).squeeze()
    return {str(i): float(probs[i]) for i in range(NUM_CLASSES)}

# ── Gradio UI ─────────────────────────────────────────────────────
with gr.Blocks(title="MNIST Digit Classifier", theme=gr.themes.Soft()) as demo: # Moved theme here

    gr.Markdown("# 🔢 MNIST Digit Classifier\nDraw, upload, or photograph a handwritten digit (0–9).")

    with gr.Tabs():

        # Tab 1: Sketchpad
        with gr.TabItem("✏️ Draw"):
            gr.Markdown("Draw a digit below. The app will automatically handle the colours.")
            with gr.Row():
                with gr.Column():
                    sketch_input = gr.Sketchpad(
                        label="Draw a digit here",
                        type="pil",
                    )
                    sketch_btn = gr.Button("Predict", variant="primary")
                with gr.Column():
                    sketch_output = gr.Label(num_top_classes=3, label="Prediction")
            # Use predict_sketch which handles the dict format + inversion
            sketch_btn.click(fn=predict_sketch, inputs=sketch_input, outputs=sketch_output)

        # Tab 2: Upload
        with gr.TabItem("📁 Upload"):
            gr.Markdown("Upload an image of a handwritten digit.")
            with gr.Row():
                with gr.Column():
                    upload_input = gr.Image(
                        type="pil",
                        label="Upload digit image",
                        sources=["upload"],
                    )
                    upload_btn = gr.Button("Predict", variant="primary")
                with gr.Column():
                    upload_output = gr.Label(num_top_classes=3, label="Prediction")
            upload_btn.click(fn=predict, inputs=upload_input, outputs=upload_output)

        # Tab 3: Camera
        with gr.TabItem("📷 Camera"):
            gr.Markdown("Take a photo of a handwritten digit.")
            with gr.Row():
                with gr.Column():
                    camera_input = gr.Image(
                        type="pil",
                        label="Camera",
                        sources=["webcam"],
                    )
                    camera_btn = gr.Button("Predict", variant="primary")
                with gr.Column():
                    camera_output = gr.Label(num_top_classes=3, label="Prediction")
            camera_btn.click(fn=predict, inputs=camera_input, outputs=camera_output)

    gr.Markdown("---\n**Tips:** Write large and centred. Good contrast between digit and background helps accuracy.")

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_2732/141843180.py:76: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="MNIST Digit Classifier", theme=gr.themes.Soft()) as demo: # Moved theme here


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da4159e2d56e53d105.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
